## Sistema de agentes musicales con LangGraph + SunnoAPI

El objetivo del siguiente proyecto es crear un sistema de agentes capaz de resolver las consultas del usuario mediante un flujo de agentes que le permita realizar tareas en función de la consulta realizada.

**Instalación de dependencias:**
```
pip install -U langgraph langchain-cohere langchain-core pydantic python-dotenv gradio ipython pillow jupyter ipywidgets

pip install -r requirements.txt
```

**Cada vez que se instale un nuevo paquete:**

```
pip freeze > requirements.txt
```

### Importaciones de paquetes iniciales:

Este es un ejemplo de las dependencias que necesitamos importar para el proyecto, pero todavía falta aclarar todos los paquetes que necesitaremos instalar por lo tanto esto sólo sirve de prueba por el momento.

In [8]:
import os
import time
import random
import requests
import gradio as gr
from typing import Annotated, List, Optional, TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.graph.message import add_messages
from dotenv import load_dotenv
from IPython.display import Image, display
from pydantic import BaseModel, Field
from langchain_cohere import ChatCohere
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, SystemMessage
from langchain_core.tools import Tool, StructuredTool, tool
from langchain_community.utilities import GoogleSerperAPIWrapper


In [2]:
#Cargamos las variables de entorno desde el archivo .env
load_dotenv(override=True)

True

### Creamos las tools + Agentes

Esto es una prueba y está sujeto a cambio lo más probable es que acabemos utilizando StructuredTool para esto.

Test de creación de tool + agente:

### Agente Investigador

In [ ]:
#Definimos el wrapper de búsqueda en Google Serper
serper = GoogleSerperAPIWrapper(serper_api_key=load_dotenv("SERPER_API_KEY"))

In [ ]:
#Definimos la tool de búsqueda web utilizando el wrapper de Serper
web_search = Tool(
    name="web_search",
    func=serper.run,
    description="Useful for when you need to answer questions about current events or find specific information on the web.",  
)

In [ ]:
#Añadimos la tool a la lista de tools del agente
tools = [web_search]    

### Agente Músico

In [9]:
# Cargamos la clave de la API de Suno desde las variables de entorno para configurar la cabecera de las solicitudes HTTP

SUNO_API_KEY=os.getenv("SUNO_API_KEY")
SUNO_CALLBACK_URL = "https://api.example.com/callback" 
SUNO_BASE_URL="https://api.sunoapi.org/api/v1"
HEADERS = {
    "Authorization": f"Bearer {SUNO_API_KEY}",
    "Content-Type": "application/json"
}

### Esquema de datos

Definimos un esquema de datos con Pydantic que se ajuste al payload que requiere Suno para hacer peticiones a su API

In [10]:
# Esquema de datos 

class MusicCustomSchema(BaseModel):
    prompt: str = Field(description="Letras de la canción o descripción detallada del contenido.")
    style: str = Field(description="Géneron musical o estilo (ej: Classical, Rock, Synthwave).")
    title: str = Field(default="Untitled", description="Título de la canción.")
    instrumental: bool = Field(default=False, description="Si es True, genera solo música sin voz.")
    negativeTags: Optional[str] = Field(default=None, description="Estilos o elementos a evitar.")
    vocalGender: Optional[str] = Field(default=None, description="Género de la voz: 'm' (masculino) o 'f' (femenino).")
    weirdnessConstraint: float = Field(default=0.0, description="Nivel de rareza experimental (0.0 a 1.0).")

### Función auxiliar

Esta función nos permite hacer peticiones GET después de ejecutar la tool de generación de audio para extraer los datos de la canción (en este caso la url)

In [5]:
def _poll_for_audio(task_id: str, max_attempts=30) -> str:
    """Consulta el estado mapeando correctamente sunoData."""
    info_url = f"{SUNO_BASE_URL}/generate/record-info?taskId={task_id}"
    
    for i in range(max_attempts):
        time.sleep(10) 
        try:
            response = requests.get(info_url, headers=HEADERS, timeout=15)
            full_res = response.json()
            data = full_res.get('data', {})
            status = data.get('status')
            
            print(f"   [Polling] Intento {i+1}: Estado {status}")

            if status == 'SUCCESS':
                # Buscamos en 'sunoData' dentro de 'response'
                response_obj = data.get('response', {})
                songs = response_obj.get('sunoData', [])
                
                if songs and len(songs) > 0:
                    # Uso el sourceAudioUrl que es el enlace directo de Suno
                    audio_url = songs[0].get('sourceAudioUrl') or songs[0].get('audioUrl')
                    return f"ESTADO: OPERACIÓN EXITOSA. URL DE DESCARGA: {audio_url}"
                else:
                    return f"ERROR: SUCCESS pero sunoData está vacío. Respuesta: {full_res}"
            
            elif status == 'FAILED':
                return f"ERROR: Fallo en Suno. {data.get('errorMessage')}"
                
        except Exception as e:
            print(f"Error en polling: {e}")
            
    return "ERROR: Tiempo de espera agotado."

### Tools del Agente Musical

In [12]:
# Tool para generación personalizada de música con Suno
@tool(args_schema=MusicCustomSchema)
def suno_generate_custom(
    prompt: str, 
    style: str, 
    title: str = "Untitled", 
    instrumental: bool = False,
    negativeTags: Optional[str] = None,
    vocalGender: Optional[str] = None,
    weirdnessConstraint: float = 0.0
) -> str:
    """Genera música personalizada usando el modelo V4_5ALL de Suno."""
    url = f"{SUNO_BASE_URL}/generate"
    
    payload = {
        "customMode": True,
        "instrumental": instrumental,
        "model": "V4_5ALL",
        "callBackUrl": SUNO_CALLBACK_URL, 
        "prompt": prompt,
        "style": style,
        "title": title,
        "styleWeight": 0.65,
        "weirdnessConstraint": weirdnessConstraint,
        "audioWeight": 0.65
    }
    
    if negativeTags:
        payload["negativeTags"] = negativeTags

    if vocalGender:
        payload["vocalGender"] = vocalGender


    try:
        print(f"DEBUG: Enviando a {url}...")
        response = requests.post(url, json=payload, headers=HEADERS, timeout=30)
        res_json = response.json()
        
        # Validación de la respuesta
        if res_json.get('code') != 200:
            msg = res_json.get('msg', 'Error desconocido')
            print(f"❌ ERROR API ({res_json.get('code')}): {msg}")
            return f"Error de Suno: {msg}"

        # Si llegamos aquí, data ya no debería ser None
        data = res_json.get('data')
        task_id = data.get('taskId')
        
        print(f"✅ Tarea aceptada. ID: {task_id}. Iniciando polling...")
        return _poll_for_audio(task_id)

    except Exception as e:
        print(f"❌ ERROR CRÍTICO: {str(e)}")
        return f"Error al llamar a Suno: {str(e)}"
    
# Esta tool hay que testearla todavía
@tool
def suno_extend_audio(audio_id: str, prompt: str) -> str:
    """Extiende una canción existente dado su audio_id."""
    url = f"{SUNO_BASE_URL}/generate/continue"
    payload = {"audioId": audio_id, "prompt": prompt}
    try:
        response = requests.post(url, json=payload, headers=HEADERS)
        task_id = response.json().get('data', {}).get('taskId')
        return _poll_for_audio(task_id)
    except Exception as e:
        return f"Error al extender: {str(e)}"

In [13]:
# 1. Definición del Estado del Agente
class AgentState(TypedDict):
    # Annotated con add_messages permite que los nuevos mensajes 
    # se concatenen al historial en lugar de sobrescribirlo.
    messages: Annotated[List[BaseMessage], add_messages]

# Nodo del Agente Musical
def musician_node(state: AgentState):
    """
    Nodo Musician Unificado:
    Combina la ingeniería de prompts detallada con la configuración
    nativa de Cohere para garantizar la entrega de la URL.
    """

    # Instrucciones elaboradas (Preamble)
    preamble = (
        "Eres el experto en producción de audio y composición AI (nodo 'mus.'). "
        "Tu responsabilidad principal es transformar las peticiones creativas del usuario "
        "en parámetros técnicos precisos para la API de Suno utilizando el modelo 'V4_5ALL'.\n\n"
        
        "### TAREAS ESPECÍFICAS:\n"
        "1. TRADUCCIÓN DE ESTILO: Mapea descripciones sensoriales a etiquetas de género en el campo 'style' "
        "(ej: 'música para estudiar' -> 'lo-fi, chill, minimalist piano').\n"
        "2. CONTROL DE RAREZA: Si el usuario menciona términos como 'experimental', 'extraño', 'vanguardista' "
        "o 'nunca antes escuchado', incrementa el valor de 'weirdnessConstraint' (rango 0.0 a 1.0).\n"
        "3. FILTRADO NEGATIVO: Identifica elementos no deseados (ej: 'sin batería', 'no guitarras') y "
        "colócalos exclusivamente en el campo 'negativeTags' sin usar palabras de negación.\n"
        "4. IDENTIDAD VOCAL: Si detectas que el usuario prefiere una voz masculina o femenina, "
        "asigna 'm' o 'f' al campo 'vocalGender'. Si pide instrumental, activa 'instrumental=True'.\n\n"
        
        "### REGLAS DE OPERACIÓN CRÍTICAS:\n"
        "- Si la herramienta devuelve 'ESTADO: OPERACIÓN EXITOSA', extrae la URL y dásela al usuario.\n"
        "- NUNCA digas que hubo un error o que lo sientes si recibes una URL de descarga en el historial.\n"
        "- No inventes la URL. Debes esperar a que la herramienta confirme el estado 'SUCCESS' mediante el polling.\n"
        "- Tu respuesta final DEBE incluir la URL del audio y un breve resumen técnico de la producción realizada.\n"
        "- IMPORTANTE: Ignora los múltiples mensajes de estado intermedio (PENDING, TEXT_SUCCESS) del historial; "
        "céntrate únicamente en el mensaje final de la herramienta que contiene la URL."
    )

    # Configuración del LLM
    llm = ChatCohere(
        model="command-a-03-2025", 
        temperature=0,
        preamble=preamble
    )
    
    # Vinculación de herramientas
    llm_with_tools = llm.bind_tools([suno_generate_custom, suno_extend_audio])

    # Invocación pasando el historial completo del estado
    response = llm_with_tools.invoke(state["messages"])
    
    return {"messages": [response]}

In [14]:
# Construcción del grafo
workflow = StateGraph(AgentState)
workflow.add_node("agente musical", musician_node)
workflow.add_node("tools", ToolNode([suno_generate_custom, suno_extend_audio]))

workflow.set_entry_point("agente musical")
workflow.add_conditional_edges("agente musical", tools_condition, ["tools", END])
workflow.add_edge("tools", "agente musical")

music_agent = workflow.compile()

### Test agente musical

In [ ]:
def run_test():
    test_query = (
        "Crea una canción de Techno melódico llamada 'Código Infinito'. "
        "Quiero que suene MUY experimental y vanguardista, pero por favor, "
        "que NO tenga sonidos de percusión agresivos ni voces. "
        "La voz debe ser femenina si decides incluir algún susurro."
    )
    
    # Inicializamos el estado con la pregunta del usuario
    inputs = {"messages": [HumanMessage(content=test_query)]}

    print(f"🔔 Iniciando Test del Agente Musical ('mus.')...\n")
    print(f"ENTRADA DEL USUARIO:\n'{test_query}'\n")
    print("-" * 50)

    last_msg = None # Variable para capturar la respuesta final

    # Ejecutamos el grafo
    for chunk in music_agent.stream(inputs, stream_mode="updates"):
        for node, values in chunk.items():
            print(f"📍 Nodo actual: {node}")
            
            # Si el nodo contiene mensajes, actualizamos last_msg
            if "messages" in values:
                last_msg = values["messages"][-1]
                
                # Opcional: Si el mensaje es una llamada a herramienta, lo indicamos
                if hasattr(last_msg, 'tool_calls') and last_msg.tool_calls:
                    for tool in last_msg.tool_calls:
                        print(f"🛠️ El agente está llamando a: {tool['name']}")

    # --- AQUÍ ESTÁ EL BLOQUE CLAVE ---
    if last_msg:
        print("\n" + "="*40)
        print("🤖 RESPUESTA FINAL DEL AGENTE:")
        print(last_msg.content)
        print("="*40)
    else:
        print("\n❌ No se recibió ninguna respuesta final del agente.")

if __name__ == "__main__":
    run_test()

### Agente Editor

In [ ]:
# Aquí continúa el código del agente editor